In [1]:
PROJECT = "C:/Users/DstMove/Desktop/claude/projects/sutlab"
WORKTREE = "C:/Users/DstMove/Desktop/claude/projects/sutlab/.claude/worktrees/jovial-euclid"

In [2]:
import pandas as pd

import sys
sys.path.insert(0, WORKTREE)

from sutlab.io import load_metadata_from_excel

In [3]:
meta = load_metadata_from_excel(
    columns_path=f"{PROJECT}/data/examples/metadata/ta_columns.xlsx",
    classifications_path=f"{PROJECT}/data/examples/metadata/ta_classifications.xlsx",
)

In [7]:
meta = load_metadata_from_excel(
    columns_path=f"{WORKTREE}/data/fixtures/metadata/columns.xlsx",
    classifications_path=f"{WORKTREE}/data/fixtures/metadata/ta_classifications.xlsx",
)

In [10]:
df = pd.read_parquet(f"{WORKTREE}/data/fixtures/ta_l_2021.parquet")

In [12]:
df

,nrnr,trans,brch,bas,ava,moms,koeb
0,A,0100,X,120,NaN,NaN,120
1,B,0100,X,80,NaN,NaN,80
2,B,0100,Y,40,NaN,NaN,40
3,C,0100,Y,120,NaN,NaN,120
4,T,0100,Z,32,NaN,NaN,32
5,A,0700,,20,NaN,NaN,20
6,B,0700,,20,NaN,NaN,20
7,C,0700,,20,NaN,NaN,20
8,A,2000,X,20,2.0,NaN,22
9,A,2000,Y,20,3.0,NaN,23


In [13]:
supply_trans = {"0100", "0700"}
supply = df[df["trans"].isin(supply_trans)]
use    = df[~df["trans"].isin(supply_trans)]

# --- Product balances (basic prices) ---
print("=== Product balances (basic prices) ===")
supply_totals = supply.groupby("nrnr")["bas"].sum()
use_totals    = use.groupby("nrnr")["bas"].sum()

for product in ["A", "B", "C"]:
    s = supply_totals[product]
    u = use_totals[product]
    print(f"  {product}: supply={s}, use={u}, balanced={s == u}")

t_supply  = supply_totals["T"]
total_ava = use["ava"].sum()
print(f"  T: supply={t_supply}, sum(ava)={total_ava}, balanced={t_supply == total_ava}")

# --- GDP at market prices ---
print("\n=== GDP at market prices ===")

final_demand_trans = {"3110", "3200", "5139", "5200", "6001"}
final_demand   = use[use["trans"].isin(final_demand_trans)]["koeb"].sum()
imports        = supply[supply["trans"] == "0700"]["bas"].sum()
gdp_expenditure = final_demand - imports
print(f"  Expenditure: final demand (purchasers') {final_demand} - imports {imports} = {gdp_expenditure}")

output        = supply[supply["trans"] == "0100"]["bas"].sum()
ic_basic      = use[use["trans"] == "2000"]["bas"].sum()
ic_ava        = use[use["trans"] == "2000"]["ava"].sum()
ic_purchasers = ic_basic + ic_ava
vat           = use["moms"].sum()
gdp_production = output - ic_purchasers + vat
print(f"  Production:  output {output} - IC (purchasers') {ic_purchasers} + VAT {vat} = {gdp_production}")

print(f"\n  Match: {gdp_expenditure == gdp_production}")

=== Product balances (basic prices) ===
  A: supply=140, use=140, balanced=True
  B: supply=140, use=140, balanced=True
  C: supply=140, use=140, balanced=True
  T: supply=32, sum(ava)=32.0, balanced=True

=== GDP at market prices ===
  Expenditure: final demand (purchasers') 342 - imports 60 = 282
  Production:  output 392 - IC (purchasers') 134.0 + VAT 24.0 = 282.0

  Match: True


In [9]:
meta.classifications.transactions

,code,name,table
0,0100,Output at basic prices,supply
1,0700,Imports,supply
2,2000,Intermediate consumption,use
3,3110,Household consumption,use
4,3200,Government collective consumption,use
5,5139,Gross fixed capital formation - other,use
6,5200,Changes in inventories,use
7,6001,Exports of domestic production,use


,nrnr,trans,brch,bas,ava,moms,koeb
0,A,0100,X,120,NaN,NaN,120
1,B,0100,X,80,NaN,NaN,80
2,B,0100,Y,40,NaN,NaN,40
3,C,0100,Y,120,NaN,NaN,120
4,T,0100,Z,32,NaN,NaN,32
5,A,0700,,20,NaN,NaN,20
6,B,0700,,20,NaN,NaN,20
7,C,0700,,20,NaN,NaN,20
8,A,2000,X,20,2.0,NaN,22
9,A,2000,Y,20,3.0,NaN,23


In [18]:
meta.classifications.transactions

,code,name,table
0,0100,Output at basic prices,supply
1,0700,Imports,supply
2,2000,Intermediate consumption,use
3,3110,Household consumption,use
4,3200,Government collective consumption,use
5,5110,Gross fixed capital formation - dwellings,use
6,6001,Exports of domestic production,use


In [ ]:
import sys

WORKTREE = "C:/Users/DstMove/Desktop/claude/projects/sutlab/.claude/worktrees/jovial-euclid"
sys.path.insert(0, WORKTREE)

from sutlab.io import load_metadata_from_excel



print("=== Columns ===")
print(meta.columns)

print("\n=== Transactions ===")
print(meta.classifications.transactions.to_string(index=False))
